Multi chan model now again lol

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from fastai.vision.all import *
from mtrain.utils import *
from mtrain.neg_mask.model.show import show_classification_report, show_confusion_matrix_using_preds
from mtrain.denorm import denormalize_imagenet
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from sklearn.model_selection import train_test_split

In [ ]:
CLEAN_PATH = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/train")
DS_PATH = CLEAN_PATH

In [ ]:
CLS_WEIGHT = torch.tensor([1.0, 2.5]).float().to("mps")


def get_learner(dls):
    learn = vision_learner(
        dls,
        xresnet18,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(CLS_WEIGHT),
        n_out=2,
        normalize=False,
        n_in=3,
        pretrained=True,
    )
    learn = learn.remove_cb(ProgressCallback)
    return learn


def get_denormalized(tens):
    image, mask = None, None
    image = denormalize_imagenet(tens)
    image = image.permute([1, 2, 0]).numpy()
    if mask is not None:
        mask = mask.numpy()
    return image, mask


def show_gradcam_for_image(
    learn, input_tensor, target_label_idx=None, layer_name="0.7.1.conv1"
):
    target_layers = [learn.model.get_submodule(layer_name)]
    img_arr, _ = get_denormalized(input_tensor[0])

    targets = [ClassifierOutputTarget(target_label_idx)]

    with GradCAM(model=learn.model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        print(img_arr.shape, grayscale_cam.shape)
        visualization = show_cam_on_image(img_arr, grayscale_cam, use_rgb=True)
        model_outputs = cam.outputs

        return visualization, img_arr, model_outputs


def show_reports(learner):
    preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
    probs, targs, decoded, losses = preds
    labels = list(BlurPadDataset.LABEL_BY_IDX.keys())
    show_classification_report(probs, targs, labels)
    show_confusion_matrix_using_preds(probs, targs, labels)
    return probs, targs, decoded, losses

In [ ]:

def get_dls(num_samples, tfm):
    image_paths = list((DS_PATH / "train").glob("*.jpg"))[:num_samples]
    stratify = [BlurPadDataset.label_func(p) for p in image_paths]
    train_paths, valid_paths = train_test_split(
        image_paths, test_size=0.2, stratify=stratify, random_state=42
    )

    train_ds = BlurPadDataset(train_paths, DS_PATH / "masks", 64, False, tfm, 15)
    valid_ds = BlurPadDataset(valid_paths, DS_PATH / "masks", 64, True, tfm, 15)
    dls = DataLoaders.from_dsets(
        train_ds,
        valid_ds,
        device=default_device(),
        num_workers=4,
        bs=16,
        # pin_memory=True,
    return dls


def vis_sample(dls, idx):
    ds = dls.train_ds
    tens, targ = ds[idx]
    print("target", targ)
    print("shape", tens.shape)
    img, _ = get_denormalized(tens)
    plt.imshow(img, cmap="gray")
    plt.show()

In [ ]:
from mtrain.neg_mask.model.datasets.blur_pad_dl import BlurPadDataset, CropTfmsOutsideBbox

def zero_overwriter(crop, crop_mask, inner_bbox):
    return CropTfmsOutsideBbox(crop, inner_bbox).overwrite_with_zeros().crop

dls = get_dls(400, zero_overwriter)

In [ ]:
vis_sample(dls, 32)

In [ ]:
dls = get_dls(20000, zero_overwriter)
learner = get_learner(dls)

In [ ]:
learner.fine_tune(1)
learner.fit_one_cycle(5)

In [ ]:
learner.fit_one_cycle(10)

In [ ]:
res = show_reports(learner)

In [ ]:
probs, targs, decoded, losses = res
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]


In [ ]:
BASELINE = mkdir(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/neg-baseline"))
torch.save(learner.model.state_dict(), BASELINE / "baseline-pad-10-crop-size-64-iter-20.pt")

In [ ]:
idx = top_loss_idxs[10]
viz, img, mo = show_gradcam_for_image(learner, learner.dls.valid_ds[idx][0].unsqueeze(0), 1, "0.7.1.convpath.1.0")
print(mo)
show([viz, img])
